# ESG 감성 변화량 분석

## 목적

`v_2`에서 생성한 기업-연도별 ESG 감성 feature를 이용해, 같은 기업의 전년 대비 ESG 감성 변화가 KCGS ESG 등급 변화와 함께 움직이는지 확인한다.

## 분석 방법

1. `v_2_sentiment_analysis_df.csv`를 불러와 `seed`와 `expanded_0_60` 사전 결과를 구분한다.
2. `dictionary_label × stock_code` 단위로 전년 값과 변화량(`delta`)을 계산한다.
3. ESG 등급 변화량(`delta_esg_grade_num`)과 감성/문장량 변화량의 Spearman 상관을 비교한다.
4. 등급 상승·유지·하락 그룹별 변화량 평균을 비교하고 Kruskal-Wallis 검정을 수행한다.
5. HC3 robust OLS로 감성 변화량과 등급 변화량의 관계를 추가 확인한다.

## 실행 결과 요약

### 데이터 구성

| 항목 | expanded_0_60 | seed |
|---|---:|---:|
| firm-year rows | 378 | 378 |
| firms | 126 | 126 |
| change rows | 252 | 252 |
| 등급 상승 | 57 | 57 |
| 등급 유지 | 142 | 142 |
| 등급 하락 | 53 | 53 |

### Spearman 변화량 분석

| feature | expanded_0_60 rho | expanded_0_60 p-value | seed rho | seed p-value | expanded - seed |
|---|---:|---:|---:|---:|---:|
| delta_esg_sentence_count | 0.177612 | 0.004684 | 0.177572 | 0.004693 | 0.000040 |
| delta_total_word_count | 0.126883 | 0.044185 | 0.126883 | 0.044185 | 0.000000 |
| delta_positive_esg_sentence_count | 0.081845 | 0.195333 | 0.080605 | 0.202209 | 0.001240 |
| delta_positive_esg_share | 0.064378 | 0.308708 | 0.068298 | 0.280108 | -0.003921 |
| delta_mean_esg_sentiment | 0.064745 | 0.305947 | 0.057912 | 0.359914 | 0.006833 |
| delta_negative_esg_sentence_count | 0.043091 | 0.495890 | 0.046692 | 0.460559 | -0.003600 |
| delta_negative_esg_share | -0.009342 | 0.882687 | 0.005288 | 0.933437 | -0.014630 |

### Kruskal-Wallis 검정

| feature | expanded_0_60 statistic | expanded_0_60 p-value | seed statistic | seed p-value |
|---|---:|---:|---:|---:|
| delta_mean_esg_sentiment | 1.196975 | 0.549642 | 0.918577 | 0.631733 |
| delta_positive_esg_share | 1.887773 | 0.389113 | 1.456828 | 0.482674 |
| delta_negative_esg_share | 3.542263 | 0.170140 | 3.465242 | 0.176820 |
| delta_esg_sentence_count | 9.612504 | 0.008178 | 9.485845 | 0.008713 |
| delta_total_word_count | 5.178174 | 0.075089 | 5.178174 | 0.075089 |

### OLS 변화량 분석

| dictionary | model | term | coef | p-value | R2 |
|---|---|---|---:|---:|---:|
| expanded_0_60 | M1_sentiment_only | delta_mean_esg_sentiment | 0.059739 | 0.252810 | 0.0060 |
| seed | M1_sentiment_only | delta_mean_esg_sentiment | 0.055836 | 0.276465 | 0.0053 |
| expanded_0_60 | M2_sentiment_plus_volume | delta_mean_esg_sentiment | 0.054021 | 0.314289 | 0.0076 |
| expanded_0_60 | M2_sentiment_plus_volume | delta_total_word_count | 0.031150 | 0.615541 | 0.0076 |
| seed | M2_sentiment_plus_volume | delta_mean_esg_sentiment | 0.050197 | 0.342545 | 0.0070 |
| seed | M2_sentiment_plus_volume | delta_total_word_count | 0.032305 | 0.602179 | 0.0070 |
| expanded_0_60 | M3_pos_neg_plus_volume | delta_positive_esg_share | 0.063564 | 0.210270 | 0.0092 |
| expanded_0_60 | M3_pos_neg_plus_volume | delta_negative_esg_share | 0.005734 | 0.918665 | 0.0092 |
| expanded_0_60 | M3_pos_neg_plus_volume | delta_total_word_count | 0.025599 | 0.687287 | 0.0092 |
| seed | M3_pos_neg_plus_volume | delta_positive_esg_share | 0.062914 | 0.214516 | 0.0092 |
| seed | M3_pos_neg_plus_volume | delta_negative_esg_share | 0.013145 | 0.805977 | 0.0092 |
| seed | M3_pos_neg_plus_volume | delta_total_word_count | 0.026085 | 0.682735 | 0.0092 |


In [1]:
from pathlib import Path
import warnings

import numpy as np
import pandas as pd
from IPython.display import display

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 100)
pd.set_option("display.max_colwidth", 160)

if Path("/content").exists():
    from google.colab import drive
    drive.mount("/content/drive")

LOCAL_ROOT = Path.cwd()
ROOT_CANDIDATES = [
    Path("/content/drive/MyDrive/UD_26"),
    Path("/content/drive/My Drive/UD_26"),
    LOCAL_ROOT,
    LOCAL_ROOT.parent,
]


def first_existing(candidates, default=None):
    for candidate in candidates:
        if candidate.exists():
            return candidate
    return default if default is not None else candidates[0]


ROOT = first_existing(
    [path for path in ROOT_CANDIDATES if (path / "final").exists() or (path / "data").exists()],
    LOCAL_ROOT,
)
FINAL_DIR = ROOT / "final"
INPUT_PATH = FINAL_DIR / "v_2_sentiment_analysis_df.csv"

print("ROOT:", ROOT)
print("INPUT_PATH:", INPUT_PATH, "OK" if INPUT_PATH.exists() else "MISSING")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
ROOT: /content/drive/MyDrive/UD_26
INPUT_PATH: /content/drive/MyDrive/UD_26/final/v_2_sentiment_analysis_df.csv OK


In [2]:
if not INPUT_PATH.exists():
    raise FileNotFoundError(
        f"{INPUT_PATH} 파일이 없습니다. 먼저 v_2.ipynb에서 analysis_df를 "
        "v_2_sentiment_analysis_df.csv로 저장하세요."
    )

analysis_df = pd.read_csv(INPUT_PATH, dtype={"stock_code": "string"}, encoding="utf-8-sig")
analysis_df["stock_code"] = analysis_df["stock_code"].astype("string").str.zfill(6)

print("loaded:", INPUT_PATH)
print("rows:", len(analysis_df))
display(analysis_df.head())


loaded: /content/drive/MyDrive/UD_26/final/v_2_sentiment_analysis_df.csv
rows: 756


,dictionary_label,stock_code,company_name,fiscal_year,esg_year,rcept_no,total_word_count,total_char_count,section_count,esg_sentence_count,positive_esg_sentence_count,negative_esg_sentence_count,neutral_esg_sentence_count,mean_esg_sentiment,positive_esg_share,negative_esg_share,neutral_esg_share,esg_sentence_per_1000_words,industry,esg_grade,e_grade,s_grade,g_grade,esg_grade_num,e_grade_num,s_grade_num,g_grade_num
0,expanded_0_60,000020,동화약품,2022,2023,20230315001100,7808,37953,3,46,2,2,42,0.013937,0.043478,0.043478,0.913043,5.891393,NaN,C,C,B,C,1,1,2,1
1,expanded_0_60,000020,동화약품,2023,2024,20240319000652,8065,38467,3,46,2,1,43,0.027842,0.043478,0.021739,0.934783,5.703658,NaN,C,B,B,C,1,2,2,1
2,expanded_0_60,000020,동화약품,2024,2025,20250318000739,8097,39259,3,50,2,0,48,0.039566,0.040000,0.000000,0.960000,6.175127,NaN,C,B,C,C,1,2,1,1
3,expanded_0_60,000040,KR모터스,2022,2023,20230322001182,4201,20136,3,17,1,0,16,0.058810,0.058824,0.000000,0.941176,4.046656,NaN,D,D,D,D,0,0,0,0
4,expanded_0_60,000040,KR모터스,2023,2024,20240321002062,4888,22402,3,14,1,1,12,0.033740,0.071429,0.071429,0.857143,2.864157,NaN,D,D,D,D,0,0,0,0


In [3]:
required_cols = {
    "dictionary_label",
    "stock_code",
    "company_name",
    "fiscal_year",
    "esg_year",
    "esg_grade_num",
    "mean_esg_sentiment",
    "positive_esg_share",
    "negative_esg_share",
    "positive_esg_sentence_count",
    "negative_esg_sentence_count",
    "esg_sentence_count",
    "total_word_count",
}

missing = sorted(required_cols - set(analysis_df.columns))
if missing:
    raise ValueError(f"필수 컬럼이 없습니다: {missing}")

use_cols = [
    "dictionary_label", "stock_code", "company_name", "fiscal_year", "esg_year", "esg_grade_num",
    "mean_esg_sentiment", "positive_esg_share", "negative_esg_share",
    "positive_esg_sentence_count", "negative_esg_sentence_count",
    "esg_sentence_count", "total_word_count",
]

panel_df = analysis_df[use_cols].copy()
for col in ["fiscal_year", "esg_year", "esg_grade_num"]:
    panel_df[col] = pd.to_numeric(panel_df[col], errors="coerce")
for col in [c for c in use_cols if c not in {"dictionary_label", "stock_code", "company_name"}]:
    panel_df[col] = pd.to_numeric(panel_df[col], errors="coerce")

panel_df = panel_df.sort_values(["dictionary_label", "stock_code", "fiscal_year"]).reset_index(drop=True)

print("panel rows:", len(panel_df))
print("firm-year rows per dictionary:")
display(panel_df.groupby("dictionary_label").agg(rows=("stock_code", "size"), firms=("stock_code", "nunique")))
display(panel_df.head())


panel rows: 756
firm-year rows per dictionary:


,rows,firms
dictionary_label,,
expanded_0_60,378,126
seed,378,126


,dictionary_label,stock_code,company_name,fiscal_year,esg_year,esg_grade_num,mean_esg_sentiment,positive_esg_share,negative_esg_share,positive_esg_sentence_count,negative_esg_sentence_count,esg_sentence_count,total_word_count
0,expanded_0_60,000020,동화약품,2022,2023,1,0.013937,0.043478,0.043478,2,2,46,7808
1,expanded_0_60,000020,동화약품,2023,2024,1,0.027842,0.043478,0.021739,2,1,46,8065
2,expanded_0_60,000020,동화약품,2024,2025,1,0.039566,0.040000,0.000000,2,0,50,8097
3,expanded_0_60,000040,KR모터스,2022,2023,0,0.058810,0.058824,0.000000,1,0,17,4201
4,expanded_0_60,000040,KR모터스,2023,2024,0,0.033740,0.071429,0.071429,1,1,14,4888


In [4]:
year_summary = (
    panel_df.groupby(["dictionary_label", "fiscal_year"])
    .agg(
        rows=("stock_code", "size"),
        firms=("stock_code", "nunique"),
        mean_grade=("esg_grade_num", "mean"),
        mean_sentiment=("mean_esg_sentiment", "mean"),
    )
    .reset_index()
)

years_per_firm = (
    panel_df.groupby(["dictionary_label", "stock_code"])["fiscal_year"]
    .nunique()
    .reset_index(name="observed_years")
    .groupby(["dictionary_label", "observed_years"])
    .size()
    .reset_index(name="firm_count")
)

print("연도별 요약")
display(year_summary)
print("기업별 관측연도 수")
display(years_per_firm)


연도별 요약


,dictionary_label,fiscal_year,rows,firms,mean_grade,mean_sentiment
0,expanded_0_60,2022,126,126,2.555556,0.044033
1,expanded_0_60,2023,126,126,2.674603,0.046776
2,expanded_0_60,2024,126,126,2.626984,0.044218
3,seed,2022,126,126,2.555556,0.044006
4,seed,2023,126,126,2.674603,0.046677
5,seed,2024,126,126,2.626984,0.044021


기업별 관측연도 수


,dictionary_label,observed_years,firm_count
0,expanded_0_60,3,126
1,seed,3,126


In [5]:
change_base_cols = [
    "mean_esg_sentiment",
    "positive_esg_share",
    "negative_esg_share",
    "positive_esg_sentence_count",
    "negative_esg_sentence_count",
    "esg_sentence_count",
    "total_word_count",
    "esg_grade_num",
]

change_df = panel_df.copy()
group_keys = ["dictionary_label", "stock_code"]

for col in change_base_cols:
    change_df[f"lag_{col}"] = change_df.groupby(group_keys)[col].shift(1)
    change_df[f"delta_{col}"] = change_df[col] - change_df[f"lag_{col}"]

change_df["year_gap"] = change_df["fiscal_year"] - change_df.groupby(group_keys)["fiscal_year"].shift(1)
change_df = change_df[change_df["year_gap"] == 1].copy()

change_df["grade_change_group"] = np.select(
    [change_df["delta_esg_grade_num"] > 0, change_df["delta_esg_grade_num"] < 0],
    ["등급 상승", "등급 하락"],
    default="등급 유지",
)

print("change rows:", len(change_df))
display(
    change_df[[
        "dictionary_label", "stock_code", "company_name", "fiscal_year",
        "esg_grade_num", "lag_esg_grade_num", "delta_esg_grade_num",
        "mean_esg_sentiment", "lag_mean_esg_sentiment",
        "delta_mean_esg_sentiment", "grade_change_group",
    ]].head(20)
)


change rows: 504


,dictionary_label,stock_code,company_name,fiscal_year,esg_grade_num,lag_esg_grade_num,delta_esg_grade_num,mean_esg_sentiment,lag_mean_esg_sentiment,delta_mean_esg_sentiment,grade_change_group
1,expanded_0_60,000020,동화약품,2023,1,1.0,0.0,0.027842,0.013937,0.013905,등급 유지
2,expanded_0_60,000020,동화약품,2024,1,1.0,0.0,0.039566,0.027842,0.011724,등급 유지
4,expanded_0_60,000040,KR모터스,2023,0,0.0,0.0,0.033740,0.058810,-0.025070,등급 유지
5,expanded_0_60,000040,KR모터스,2024,0,0.0,0.0,0.055080,0.033740,0.021340,등급 유지
7,expanded_0_60,000050,경방,2023,1,1.0,0.0,0.029300,0.032491,-0.003191,등급 유지
8,expanded_0_60,000050,경방,2024,1,1.0,0.0,-0.000289,0.029300,-0.029589,등급 유지
10,expanded_0_60,000070,삼양홀딩스,2023,4,3.0,1.0,0.108463,0.009919,0.098544,등급 상승
11,expanded_0_60,000070,삼양홀딩스,2024,3,4.0,-1.0,0.121063,0.108463,0.012599,등급 하락
13,expanded_0_60,000080,하이트진로,2023,3,2.0,1.0,-0.017874,0.021867,-0.039741,등급 상승
14,expanded_0_60,000080,하이트진로,2024,2,3.0,-1.0,0.025882,-0.017874,0.043756,등급 하락


In [6]:
change_summary = (
    change_df.groupby(["dictionary_label", "fiscal_year", "grade_change_group"])
    .size()
    .rename("row_count")
    .reset_index()
)

overall_change_summary = (
    change_df.groupby(["dictionary_label", "grade_change_group"])
    .size()
    .rename("row_count")
    .reset_index()
)

print("전체 등급 변화 그룹 분포")
display(overall_change_summary)
print("연도별 등급 변화 그룹 분포")
display(change_summary)


전체 등급 변화 그룹 분포


,dictionary_label,grade_change_group,row_count
0,expanded_0_60,등급 상승,57
1,expanded_0_60,등급 유지,142
2,expanded_0_60,등급 하락,53
3,seed,등급 상승,57
4,seed,등급 유지,142
5,seed,등급 하락,53


연도별 등급 변화 그룹 분포


,dictionary_label,fiscal_year,grade_change_group,row_count
0,expanded_0_60,2023,등급 상승,34
1,expanded_0_60,2023,등급 유지,69
2,expanded_0_60,2023,등급 하락,23
3,expanded_0_60,2024,등급 상승,23
4,expanded_0_60,2024,등급 유지,73
5,expanded_0_60,2024,등급 하락,30
6,seed,2023,등급 상승,34
7,seed,2023,등급 유지,69
8,seed,2023,등급 하락,23
9,seed,2024,등급 상승,23


In [7]:
from scipy.stats import spearmanr

text_delta_cols = [
    "delta_mean_esg_sentiment",
    "delta_positive_esg_share",
    "delta_negative_esg_share",
    "delta_positive_esg_sentence_count",
    "delta_negative_esg_sentence_count",
    "delta_esg_sentence_count",
    "delta_total_word_count",
]


def spearman_delta_table(data, y_col, x_cols):
    rows = []
    for dictionary_label, group in data.groupby("dictionary_label"):
        for x_col in x_cols:
            tmp = group[[y_col, x_col]].replace([np.inf, -np.inf], np.nan).dropna()
            if len(tmp) < 3 or tmp[y_col].nunique() < 2 or tmp[x_col].nunique() < 2:
                rho, pvalue = np.nan, np.nan
            else:
                rho, pvalue = spearmanr(tmp[y_col], tmp[x_col])
            rows.append({
                "dictionary_label": dictionary_label,
                "feature": x_col,
                "target": y_col,
                "n": len(tmp),
                "spearman_rho": rho,
                "p_value": pvalue,
            })
    return pd.DataFrame(rows).sort_values(["feature", "spearman_rho"], ascending=[True, False])


spearman_change_df = spearman_delta_table(change_df, "delta_esg_grade_num", text_delta_cols)
display(spearman_change_df)

if {"seed", "expanded_0_60"}.issubset(set(spearman_change_df["dictionary_label"])):
    delta_compare_df = (
        spearman_change_df
        .pivot(index="feature", columns="dictionary_label", values="spearman_rho")
        .reset_index()
    )
    delta_compare_df["rho_expanded_minus_seed"] = delta_compare_df["expanded_0_60"] - delta_compare_df["seed"]
    display(delta_compare_df.sort_values("rho_expanded_minus_seed", ascending=False))


,dictionary_label,feature,target,n,spearman_rho,p_value
5,expanded_0_60,delta_esg_sentence_count,delta_esg_grade_num,252,0.177612,0.004684
12,seed,delta_esg_sentence_count,delta_esg_grade_num,252,0.177572,0.004693
0,expanded_0_60,delta_mean_esg_sentiment,delta_esg_grade_num,252,0.064745,0.305947
7,seed,delta_mean_esg_sentiment,delta_esg_grade_num,252,0.057912,0.359914
11,seed,delta_negative_esg_sentence_count,delta_esg_grade_num,252,0.046692,0.460559
4,expanded_0_60,delta_negative_esg_sentence_count,delta_esg_grade_num,252,0.043091,0.495890
9,seed,delta_negative_esg_share,delta_esg_grade_num,252,0.005288,0.933437
2,expanded_0_60,delta_negative_esg_share,delta_esg_grade_num,252,-0.009342,0.882687
3,expanded_0_60,delta_positive_esg_sentence_count,delta_esg_grade_num,252,0.081845,0.195333
10,seed,delta_positive_esg_sentence_count,delta_esg_grade_num,252,0.080605,0.202209


dictionary_label,feature,expanded_0_60,seed,rho_expanded_minus_seed
1,delta_mean_esg_sentiment,0.064745,0.057912,0.006833
4,delta_positive_esg_sentence_count,0.081845,0.080605,0.001240
0,delta_esg_sentence_count,0.177612,0.177572,0.000040
6,delta_total_word_count,0.126883,0.126883,0.000000
2,delta_negative_esg_sentence_count,0.043091,0.046692,-0.003600
5,delta_positive_esg_share,0.064378,0.068298,-0.003921
3,delta_negative_esg_share,-0.009342,0.005288,-0.014630


In [8]:
group_cols = [
    "delta_mean_esg_sentiment",
    "delta_positive_esg_share",
    "delta_negative_esg_share",
    "delta_esg_sentence_count",
    "delta_total_word_count",
]

group_summary = (
    change_df.groupby(["dictionary_label", "grade_change_group"])[group_cols]
    .agg(["count", "mean", "median", "std"])
)

display(group_summary)


delta_mean_esg_sentiment            \
                                                       count      mean   
dictionary_label grade_change_group                                      
expanded_0_60    등급 상승                                    57  0.005183   
                 등급 유지                                   142 -0.000831   
                 등급 하락                                    53 -0.002909   
seed             등급 상승                                    57  0.004925   
                 등급 유지                                   142 -0.001091   
                 등급 하락                                    53 -0.002340   

                                                         \
                                       median       std   
dictionary_label grade_change_group                       
expanded_0_60    등급 상승               0.000000  0.035542   
                 등급 유지               0.000778  0.026122   
                 등급 하락              -0.005605  0.030184   
seed             등급 상승              -0.000079  0.035825   
                 등급 유지               0.000617  0.026478   
                 등급 하락              -0.005993  0.030648   

                                    delta_positive_esg_share            \
                                                       count      mean   
dictionary_label grade_change_group                                      
expanded_0_60    등급 상승                                    57  0.004170   
                 등급 유지                                   142 -0.000324   
                 등급 하락                                    53 -0.005367   
seed             등급 상승                                    57  0.004294   
                 등급 유지                                   142 -0.000765   
                 등급 하락                                    53 -0.004786   

                                                         \
                                       median       std   
dictionary_label grade_change_group                       
expanded_0_60    등급 상승              -0.001373  0.037215   
                 등급 유지               0.000000  0.022872   
                 등급 하락              -0.007087  0.025843   
seed             등급 상승               0.000000  0.038000   
                 등급 유지               0.000000  0.023454   
                 등급 하락              -0.007071  0.026203   

                                    delta_negative_esg_share            \
                                                       count      mean   
dictionary_label grade_change_group                                      
expanded_0_60    등급 상승                                    57 -0.002280   
                 등급 유지                                   142  0.001035   
                 등급 하락                                    53 -0.003146   
seed             등급 상승                                    57 -0.001850   
                 등급 유지                                   142  0.000903   
                 등급 하락                                    53 -0.003166   

                                                         \
                                       median       std   
dictionary_label grade_change_group                       
expanded_0_60    등급 상승              -0.000875  0.015818   
                 등급 유지               0.000000  0.017867   
                 등급 하락              -0.000806  0.017866   
seed             등급 상승              -0.000649  0.015498   
                 등급 유지               0.000000  0.017814   
                 등급 하락              -0.000809  0.017332   

                                    delta_esg_sentence_count                   \
                                                       count      mean median   
dictionary_label grade_change_group                                             
expanded_0_60    등급 상승                                    57  4.403509    5.0   
                 등급 유지                                   142  4.373239    4.0   
                 등급 하락            

In [9]:
from scipy.stats import kruskal

test_rows = []
for dictionary_label, dictionary_group in change_df.groupby("dictionary_label"):
    for col in group_cols:
        groups = [
            g[col].replace([np.inf, -np.inf], np.nan).dropna().values
            for _, g in dictionary_group.groupby("grade_change_group")
        ]
        groups = [g for g in groups if len(g) > 0]
        if len(groups) >= 2 and sum(len(g) for g in groups) >= 3:
            stat, pvalue = kruskal(*groups)
        else:
            stat, pvalue = np.nan, np.nan
        test_rows.append({
            "dictionary_label": dictionary_label,
            "feature": col,
            "test": "Kruskal-Wallis",
            "statistic": stat,
            "p_value": pvalue,
        })

kruskal_df = pd.DataFrame(test_rows)
display(kruskal_df)


,dictionary_label,feature,test,statistic,p_value
0,expanded_0_60,delta_mean_esg_sentiment,Kruskal-Wallis,1.196975,0.549642
1,expanded_0_60,delta_positive_esg_share,Kruskal-Wallis,1.887773,0.389113
2,expanded_0_60,delta_negative_esg_share,Kruskal-Wallis,3.542263,0.170140
3,expanded_0_60,delta_esg_sentence_count,Kruskal-Wallis,9.612504,0.008178
4,expanded_0_60,delta_total_word_count,Kruskal-Wallis,5.178174,0.075089
5,seed,delta_mean_esg_sentiment,Kruskal-Wallis,0.918577,0.631733
6,seed,delta_positive_esg_share,Kruskal-Wallis,1.456828,0.482674
7,seed,delta_negative_esg_share,Kruskal-Wallis,3.465242,0.176820
8,seed,delta_esg_sentence_count,Kruskal-Wallis,9.485845,0.008713
9,seed,delta_total_word_count,Kruskal-Wallis,5.178174,0.075089


In [10]:
import statsmodels.api as sm


def zscore(series):
    series = series.astype(float)
    std = series.std(ddof=0)
    if std == 0 or pd.isna(std):
        return series * 0
    return (series - series.mean()) / std


def robust_delta_ols(data, y_col, x_cols):
    reg_df = data[[y_col] + x_cols].replace([np.inf, -np.inf], np.nan).dropna().copy()
    if len(reg_df) < len(x_cols) + 3:
        raise ValueError(f"too few complete rows after dropna: n={len(reg_df)}, predictors={len(x_cols)}")
    y = reg_df[y_col].astype(float)
    X = reg_df[x_cols].apply(zscore)
    X = sm.add_constant(X, has_constant="add")
    model = sm.OLS(y, X).fit(cov_type="HC3")
    result = pd.DataFrame({
        "term": model.params.index,
        "coef": model.params.values,
        "std_err_HC3": model.bse.values,
        "t": model.tvalues.values,
        "p_value": model.pvalues.values,
    })
    return model, result, reg_df


model_specs = {
    "M1_sentiment_only": ["delta_mean_esg_sentiment"],
    "M2_sentiment_plus_volume": ["delta_mean_esg_sentiment", "delta_total_word_count"],
    "M3_pos_neg_plus_volume": ["delta_positive_esg_share", "delta_negative_esg_share", "delta_total_word_count"],
}

ols_rows = []
for dictionary_label, dictionary_group in change_df.groupby("dictionary_label"):
    print("\n" + "=" * 80)
    print("dictionary_label:", dictionary_label)
    for name, x_cols in model_specs.items():
        try:
            model, result, reg_df = robust_delta_ols(dictionary_group, "delta_esg_grade_num", x_cols)
            print("\n" + name, "n=", len(reg_df), "R2=", round(model.rsquared, 4))
            display(result)
            for _, row in result.iterrows():
                ols_rows.append({
                    "dictionary_label": dictionary_label,
                    "model": name,
                    "n": len(reg_df),
                    "r2": model.rsquared,
                    **row.to_dict(),
                })
        except Exception as exc:
            print(f"{name} skipped:", exc)

ols_results_df = pd.DataFrame(ols_rows)



dictionary_label: expanded_0_60

M1_sentiment_only n= 252 R2= 0.006


,term,coef,std_err_HC3,t,p_value
0,const,0.035714,0.048654,0.734051,0.462918
1,delta_mean_esg_sentiment,0.059739,0.052240,1.143551,0.252810



M2_sentiment_plus_volume n= 252 R2= 0.0076


,term,coef,std_err_HC3,t,p_value
0,const,0.035714,0.048907,0.730243,0.465242
1,delta_mean_esg_sentiment,0.054021,0.053684,1.006264,0.314289
2,delta_total_word_count,0.031150,0.062030,0.502180,0.615541



M3_pos_neg_plus_volume n= 252 R2= 0.0092


...

,term,coef,std_err_HC3,t,p_value
0,const,0.035714,0.048665,0.733887,0.463018
1,delta_mean_esg_sentiment,0.055836,0.051306,1.088294,0.276465



M2_sentiment_plus_volume n= 252 R2= 0.007


,term,coef,std_err_HC3,t,p_value
0,const,0.035714,0.048916,0.730121,0.465317
1,delta_mean_esg_sentiment,0.050197,0.052887,0.949149,0.342545
2,delta_total_word_count,0.032305,0.061973,0.521269,0.602179



M3_pos_neg_plus_volume n= 252 R2= 0.0092


,term,coef,std_err_HC3,t,p_value
0,const,0.035714,0.049074,0.727764,0.466758
1,delta_positive_esg_share,0.062914,0.050686,1.241243,0.214516
2,delta_negative_esg_share,0.013145,0.053520,0.245620,0.805977
3,delta_total_word_count,0.026085,0.063818,0.408733,0.682735


In [11]:
if not spearman_change_df.empty:
    top = (
        spearman_change_df
        .replace([np.inf, -np.inf], np.nan)
        .dropna(subset=["spearman_rho"])
        .assign(abs_rho=lambda df: df["spearman_rho"].abs())
        .sort_values("abs_rho", ascending=False)
        .iloc[0]
    )
    print("변화량 분석 요약:")
    print(
        f"{top['dictionary_label']} 사전 기준에서 {top['feature']}와 "
        f"delta_esg_grade_num의 Spearman rho={top['spearman_rho']:.3f}, "
        f"p={top['p_value']:.3f}입니다."
    )


변화량 분석 요약:
expanded_0_60 사전 기준에서 delta_esg_sentence_count와 delta_esg_grade_num의 Spearman rho=0.178, p=0.005입니다.
